# Quantization Aware Training + Knowledge Distillation Benchmarking

In [1]:
import torch
import torch.onnx
from torch.ao.quantization.quantize_fx import convert_fx

from src.utils import load_data
from src.Quantization.utils.model_setup import setup_qat_student_model, quantization_mode
from src.utils import benchmark
from src.utils.model_setup import setup_model
from src.utils import test_inference

 You need to install pymongo>=3.9.0 in order to use MongoOutput 


### Load Original and Quantized model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/quantized_student_state.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_qat_student_model(model_name="mobilenet_v2",num_classes=num_classes)

example_inputs = next(iter(dataloaders["train"]))[0].to(device)
student_model = quantization_mode(model, "fx", example_inputs=example_inputs)

# Move the model to CPU if needed (conversion is typically done on CPU).
student_model = student_model.to("cpu")

quantized_model = convert_fx(student_model)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
quantized_model.load_state_dict(state_dict)

# Set to eval mode.
quantized_model.eval()

teacher_model = setup_model("mobilenet_v2", None, num_classes)

Model prepared using FX Graph Mode QAT.


/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/utils.py:407: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/_utils.py:392: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


### Perform Benchmarking (model_size, inference time, throughput, memory usage)

In [ ]:
device = torch.device("cpu")

benchmark(model1=teacher_model, model2=quantized_model, dataloader=dataloaders["test"], device=device)

In [ ]:
test_inference(quantized_model, dataloaders["test"], device, "models")

In [ ]:
def export_model_to_onnx(model: torch.nn.Module, example_input: torch.Tensor, onnx_file_path: str, opset_version: int = 12) -> None:
    """
    Exports a given PyTorch model to the ONNX format.

    This function uses torch.onnx.export to convert the model to ONNX. It assumes the model is in eval mode.
    Note that quantized models may face compatibility issues with ONNX; ensure that the opset_version and model
    configuration are supported.

    Args:
        model (torch.nn.Module): The PyTorch model to export.
        example_input (torch.Tensor): An example input tensor with the appropriate shape.
        onnx_file_path (str): Path where the ONNX file will be saved.
        opset_version (int): ONNX opset version to use. Defaults to 12.
    """
    # Ensure the model is in evaluation mode.
    model.eval()

    # Export the model.
    torch.onnx.export(
        model,                          # model being exported
        example_input,                  # example input to the model
        onnx_file_path,                 # where to save the ONNX model
        export_params=True,             # store the trained parameter weights inside the model file
        opset_version=opset_version,    # specify the ONNX version to export the model to
        do_constant_folding=True,       # execute constant folding for optimization
        input_names=['input'],          # the model's input names
        output_names=['output'],        # the model's output names
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}  # enable dynamic batch size
    )
    print(f"Model successfully exported to {onnx_file_path}")

device = torch.device("cpu")
# Example usage:
# Given your code, you have quantized_model and example_inputs already defined.
onnx_export_path: str = "quantized_student.onnx"
export_model_to_onnx(quantized_model, example_inputs.to(device), onnx_export_path)

Traceback (most recent call last):
  File "/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph_module.py", line 348, in __call__
    return super(self.cls, obj).__call__(*args, **kwargs)  # type: ignore[misc]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1726, in _slow_forward
    result = self.forward(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<eval_with_key>.5", line 7, in forw

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument scale in method wrapper_CUDA_tensor_qparams_quantize_per_tensor)